# MediConfirm
Check a medication package against a saved list, ask questions, and record an explicit intake self-report.

This classroom demo uses empty medication packages. Select a schedule manually. Use localhost or HTTPS and allow browser camera and microphone access.

## 1. Install dependencies
Run this cell once in your Anaconda kernel. Restart the kernel after installation, then continue with section 2. Whisper may download its model on first use.

In [ ]:
import sys
import subprocess
from pathlib import Path

PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "src" / "mediconfirm").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from the mediconfirm_v2 folder.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")], cwd=PROJECT_DIR)
print("Installation complete. Restart the kernel and continue with section 2.")

## 2. Load API configuration
The existing key and model are stored in `config.local.json`, which is excluded from Git. No key entry is needed for each run. On a different computer, copy `config.example.json` to `config.local.json` and fill in the key once. The `HKBU_API_KEY` environment variable can override the local key.

In [ ]:
import json
import os
import sys
from pathlib import Path

PROJECT_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "src" / "mediconfirm").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open this notebook from the mediconfirm_v2 folder.")
if str(PROJECT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / "src"))
config_path = PROJECT_DIR / "config.local.json"
config = json.loads(config_path.read_text()) if config_path.exists() else {}
API_KEY = os.environ.get("HKBU_API_KEY", config.get("api_key", "")).strip()
MODEL_NAME = config.get("model", "gpt-4.1")
API_VERSION = config.get("api_version", "2025-04-01-preview")
if not API_KEY or API_KEY.startswith("YOUR_"):
    raise ValueError("Set your key once in config.local.json or HKBU_API_KEY.")
print("API configuration loaded. Model:", MODEL_NAME)

## 3. Medication list
Enter each medication's name, time, strength and dose. Use **+ Add medication** for additional rows, then **Save medication list**. Times use `08:00`; multiple daily times use `08:00;20:00`. Strength means the amount per tablet, such as `50 mg`; dose means the quantity per intake, such as `1 tablet`.

The current saved list is included in this local copy. Future changes are saved only to this project's `data/personal_medications.json`.

In [ ]:
import importlib
from IPython.display import display
from mediconfirm import bulk_medications, medication_form

importlib.reload(bulk_medications)
importlib.reload(medication_form)
if "medication_form_widget" in globals():
    medication_form_widget.close()
medication_form_widget = medication_form.MedicationListForm(PROJECT_DIR / "data" / "personal_medications.json")
display(medication_form_widget.widget)

## 4. Run the assistant
Select the schedule, then **Start medication check**. Capture one package. A **Match** enables explicit intake confirmation; **Mismatch** explains the difference; **Uncertain** asks for a clearer photo.

You can type or record English, review the transcript, and continue a conversation. After a matching photo, use the confirmation buttons or submit reviewed status text. Chat messages and transcription alone never create Taken records.

Under **Demo reminders and family notifications**, choose **New demo (separate log)** and **Start schedule reminder**. Advance 5 simulated minutes for a repeat question; advance another 5 for Unconfirmed and a simulated notification. For a separate severe-symptom test, start a new demo and submit `I have chest pain` in the conversation. The routine flow pauses and a simulated notification appears. No real message is sent.

Retake, replay, edit, ask follow-up questions, or cancel as needed. A cancelled session keeps records already saved.

In [ ]:
from IPython.display import display
import importlib
import ipywidgets as widgets
from mediconfirm import integrations, simple_service, drug_sources, dialogue, speech_tools, intake_store, plan_actions, simple_ui

if "_schedule_subscription" in globals():
    _schedule_subscription()
if globals().get("simple_app") is not None:
    simple_app.close()
if globals().get("time_selector") is not None:
    time_selector.close()
if "_schedule_panel" in globals():
    _schedule_panel.close()
for module in (integrations, simple_service, drug_sources, dialogue, speech_tools, intake_store, plan_actions, simple_ui):
    importlib.reload(module)
if "medication_form_widget" not in globals():
    raise ValueError("Run section 3 first.")

CONFIRMATION_TIMEOUT_SECONDS = 10 * 60
simple_app = None
time_selector = None
_schedule_panel = widgets.VBox()

def sync_saved_medication_list(medications):
    global simple_app, time_selector
    if not medications:
        _schedule_panel.children = (widgets.HTML("Add and save at least one medication above."),)
        return
    times = sorted({med["time"] for med in medications})
    if simple_app is None:
        time_selector = widgets.Dropdown(options=times, description="Schedule:")
        simple_app = simple_ui.SimpleCameraApp(
            api_key=API_KEY, model=MODEL_NAME, api_version=API_VERSION,
            medications=medications, selected_time=time_selector.value, voice=True,
            log_path=PROJECT_DIR / "data" / "intake_records.json",
            confirmation_timeout_seconds=CONFIRMATION_TIMEOUT_SECONDS,
        )
        def update_check_time(change):
            if change["new"] is not None:
                simple_app.set_check_time(change["new"])
        time_selector.observe(update_check_time, names="value")
        _schedule_panel.children = (time_selector, simple_app)
    else:
        previous_time = time_selector.value
        simple_app.medications = medications
        time_selector.options = times
        time_selector.value = previous_time if previous_time in times else times[0]
    simple_app.set_check_time(time_selector.value)

_schedule_subscription = medication_form_widget.on_saved(sync_saved_medication_list)
sync_saved_medication_list(medication_form_widget.medications)
display(_schedule_panel)

## Scope

Records are self-reports. Unconfirmed does not establish a missed dose. Severe-symptom handling covers predefined English phrases, not medical triage. General-use explanations require a verified DailyMed source. Browser polling stops when the page or kernel closes; deadlines are reconciled on reopening. The prototype does not diagnose, change prescriptions, send real messages, or run as a phone app.

## 5. Try the phone interface
Run `python run_mobile.py` from this folder, or open `Start_Mobile.command` on macOS. Keep the terminal open. Connect the phone to the same Wi-Fi and open the printed Phone link.

Use **Take a photo** or **Choose an image**. The API key stays on the computer. Phone sessions have separate lists and logs under `data/mobile_runs/`. General uses are backed by dated official DailyMed label snapshots for the demo medicines, with online lookup for other matches.

On a plain HTTP phone connection, use typing or upload a short recording. Live browser microphone access requires HTTPS. This is a local test interface, not a publicly hosted app.
